In [1]:
# Cell 1: 导入库
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import os

In [46]:
# Cell 2: 定义简单的工具
@tool
def add(a: float, b: float) -> float:
    """计算两个数的和"""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """计算两个数的乘积"""
    return a * b

tools = [add, multiply]

In [47]:
# Cell 3: 配置 LLM（用 DeepSeek，兼容 OpenAI 接口）
llm = ChatOpenAI(
    model="deepseek-chat",
    openai_api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com",
    temperature=0
)

In [48]:
# Cell 4: 创建 Agent
prompt = """你是一个有帮助的助手。你必须使用以下工具：

{tools}

工具名称：{tool_names}

规则：不要自己计算，一定要调用工具

问题：{input}

思考过程：{agent_scratchpad}
"""

agent = create_agent(model=llm, tools=tools, system_prompt=prompt)

In [50]:
# Cell 5: 测试 Agent
# 这个输出结果是不正确的，可能是工具过于简单导致模型使用了默认的prompt。
result = agent.invoke({"input": "请计算123和456的和。"})
print(result['messages'][-1].content)

计算结果是 77。

所以，(3+4) × (5+6) = 7 × 11 = 77


In [51]:
# print(agent.get_prompts())
# print(result)

In [6]:
#print(result)
#print(type(result))
#print(list(result.keys()))
#print(list(result.values()))
#print(len(result['messages']))
#print(type(result['messages'][-1]))
#print(result['messages'][-1].content)

In [7]:
# 定义一个搜索工具（模拟）
@tool
def search(query: str) -> str:
    """搜索信息（模拟）"""
    if "天气" in query:
        return "北京今天晴天，25度"
    return f"未找到关于 '{query}' 的信息"

tools_new = [search, add, multiply]

In [19]:
# Cell 4: 创建 Agent
prompt_new = """你是一个有帮助的助手。你可以使用以下工具：

{tools}

工具名称：{tool_names}

规则：
- 请使用并且仅用相应的工具。如果工具无法提供正确的答案请说“不知道”。
- 不要使用 add 或 multiply 工具来处理天气问题。
- search 工具是你唯一能获取天气信息的方式

问题：{input}

思考过程：{agent_scratchpad}
"""

agent_new = create_agent(model=llm, tools=tools_new, system_prompt=prompt_new)

In [20]:
result_new = agent_new.invoke({"input": "今天的天气如何？"})
print(result_new['messages'][-1].content)

根据搜索结果，今天北京的天气是晴天，气温25度。


In [21]:
#print(agent_new)
#print(result_new['messages'])

## 基础理解（应该能立即回答）
### 用你自己的话解释：Agent 和 RAG 的核心区别是什么？
- agent是llm+工具+prompt，没有提前训练的rag vector的过程。rag是需要预训练的。
- 补充：Agent 和 RAG 的核心区别

维度	RAG	Agent
核心	检索 + 生成	推理 + 行动
工具	只有检索器	多种工具（计算、搜索、API 调用等）
流程	固定：检索 → 生成	动态：思考 → 行动 → 观察 → 循环
典型场景	问答、文档摘要	自动化任务、多步推理
你的回答"Agent 没有 RAG 的预训练过程"不够准确——两者都不需要预训练，都是基于已有 LLM 的。
### Agent 中的 "Tool" 是什么？你今天的 demo 里定义了哪几个 Tool？
- tool是外接给llm可以使用的工具，llm可以根据输入来决定使用。我今天定义了add和multiply，还有后边的search。
### create_tool_calling_agent 和 create_react_agent 有什么不同？哪个更适合简单任务？
- 在langchain1.0中已经不做区分了吧。
- 补充：两个 Agent 创建方式的区别

create_tool_calling_agent	create_react_agent
依赖	API 原生支持 tool calling	不依赖，纯 prompt 驱动
适用	OpenAI、DeepSeek（支持 function calling）	任何 LLM
稳定性	高	中（依赖 prompt 质量）
简单任务	✅ 更适合	也可以
## 动手验证（需要运行代码或改参数）
### 给 Agent 增加一个 subtract（减法）工具，让它能计算 10 - 3。需要改几行代码？
- 需要添加工具定义，再加上修改tools集。
### 把 Agent 的 verbose=False，输出会有什么变化？这个参数的作用是什么？
- 会让输出的result里的内容减少。
### 问 Agent 一个需要多步计算的问题，比如 (3 + 4) * 2，它能正确回答吗？为什么？
- 应该可以，llm很可能会根据自己的理解绕开工具使用自己的方法去计算。
## 深入思考（面试可能会问）
### 如果 Agent 调用 Tool 时传入了错误的参数类型（比如把字符串传给需要数字的加法函数），会发生什么？如何避免？
- tool会报错，导致llm无法输出或者程序卡住。最好在tools层面或者llm层面定义输入类型并且检查。
### Agent 有可能陷入无限循环（比如反复调用同一个 Tool 不停止），你会怎么设计机制来防止？
- 在prompt里定义一个最大次数，或者修改tools的定义。
- LangChain 的 AgentExecutor 有 max_iterations 参数
### 你今天的 Agent 只用到了计算工具。如果让它能查天气、发邮件、搜索网页，架构上需要做什么扩展？
- 需要在tools中调用其他的api来获取数据并进行处理。

In [57]:
# 使用langchain_classic中的create_tool_calling_agent来尝试回答前面输出错误的求和
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [58]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有帮助的助手。你可以使用提供的工具来回答问题。"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# 创建 Agent
agent = create_tool_calling_agent(llm, tools, prompt)

# 创建 AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

# 测试
result = agent_executor.invoke({"input": "123和456的和是多少？"})
print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `add` with `{'a': 123, 'b': 456}`
responded: 我来帮您计算123和456的和。

579.0123和456的和是579。

> Finished chain.
123和456的和是579。
